# 모의고사 06. 유방암 진단 (이진분류)

> 실제 시험처럼 이론 설명 없이 문제만 순서대로 제시합니다. **90분 타이머를 맞추고** 풀어보세요. 오픈북 허용 문서(numpy/pandas/matplotlib/seaborn/scikit-learn/tensorflow/XGBoost 공식문서)만 참고할 수 있다고 가정합니다.

### 데이터 소개
`data/breast_cancer.csv` - 세포 조직 검사 수치(반지름, 질감 등 30개 특성)로 양성(1)/악성(0)을 진단합니다. 의료 데이터에서는 특히 **재현율(recall)**이 중요합니다 - 악성을 양성으로 놓치면 안 되기 때문입니다.

> ⏱️ **[실전 타임어택 가이드]** 데이터 탐색 10분 ➔ 전처리 20분 ➔ 머신러닝 30분 ➔ 딥러닝 30분
> 막히는 부분은 주석으로 남기고 다음 문제로 넘어가는 것이 합격 전략입니다.


## 목차
1교시 데이터 로딩 & EDA · 2교시 데이터 시각화 · 3교시 데이터 전처리 · 4교시 머신러닝 모델링 · 5교시 딥러닝 모델링 · 채점 가이드

In [ ]:
import sys
sys.path.insert(0, '../00_시작하기')
import aice_grader as grader

# 실전처럼 시간 제한(90분)을 두고 풀어보세요. 아래 셀을 실행하면 타이머가 시작됩니다.
timer = grader.ExamTimer(minutes=90)
timer.start()

## 1교시: 데이터 로딩 & EDA

**문제 1.** `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`을 각각 `pd`, `np`, `plt`, `sns`로 불러오고, `../data/breast_cancer.csv`을 `df`로 불러와 shape을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/breast_cancer.csv')
df.head()
print(df.shape)
```

</details>

**문제 2.** `df`의 shape과 `target` 분포를 확인하세요. (target: 0=악성, 1=양성)

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(df.shape)
print(df['target'].value_counts())  # 악성/양성 클래스 개수를 확인해 불균형 여부를 미리 체크  # 범주형 데이터의 항목별 개수를 세어 데이터 분포 확인
```

</details>

**문제 3.** `target`과 상관관계가 가장 강한 변수 5개를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
corr = df.corr()['target'].drop('target')  # 수치형 변수 간의 상관계수를 계산하여 서로의 연관성 파악
print(corr.abs().sort_values(ascending=False).head())
```

</details>

## 2교시: 데이터 시각화

**문제 4.** `mean radius`를 `target`별로 histplot(hue='target')으로 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.histplot(data=df, x='mean radius', hue='target', kde=True)  # hue로 악성/양성 그룹을 나눠 분포가 얼마나 겹치는지 비교
plt.show()
```

</details>

**문제 5.** `mean radius`, `mean texture`, `mean area`, `target` 네 컬럼만 상관관계 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
cols = ['mean radius', 'mean texture', 'mean area', 'target']
sns.heatmap(df[cols].corr(), annot=True, cmap='coolwarm')  # 수치형 변수 간의 상관계수를 계산하여 서로의 연관성 파악
plt.show()
```

</details>

## 3교시: 데이터 전처리

**문제 6.** `train_test_split(test_size=0.2, stratify=y, random_state=42)`로 분할하고 `StandardScaler`를 적용하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # 데이터를 모델 학습용(train)과 검증용(test)으로 분리
scaler = StandardScaler()  # 데이터를 평균 0, 표준편차 1이 되도록 스케일링 (거리 기반, 딥러닝 알고리즘에 필수)
X_train_s = scaler.fit_transform(X_train)  # 훈련 데이터로 스케일링 기준을 학습(fit)하고 변환(transform)까지 수행
X_test_s = scaler.transform(X_test)  # 훈련 데이터에서 학습된 기준을 그대로 적용하여 평가 데이터를 변환 (데이터 누수 방지)
```

</details>

## 4교시: 머신러닝 모델링

**문제 7.** `LogisticRegression(max_iter=5000)`을 학습시키고 accuracy, recall, roc_auc를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
logi = LogisticRegression(max_iter=5000)  # 주로 이진 분류에 쓰이는 로지스틱 회귀 모델 생성
logi.fit(X_train_s, y_train)
pred = logi.predict(X_test_s)
proba = logi.predict_proba(X_test_s)[:, 1]
print(accuracy_score(y_test, pred))  # 전체 예측 중 올바르게 맞춘 비율(정확도) 계산
print(recall_score(y_test, pred))
print(roc_auc_score(y_test, proba))
```

</details>

**문제 8.** ROC curve를 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, proba)  # thresholds는 여기서 쓰지 않으므로 _로 무시
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'k--')  # 대각선(랜덤 추측 수준)과 비교해 곡선이 왼쪽 위로 많이 벗어날수록 좋은 모델
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.show()
```

</details>

## 5교시: 딥러닝 모델링

**문제 9.** 출력층 1노드(sigmoid)인 DL 모델을 만들어 학습시키고 recall을 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
model = Sequential()  # 딥러닝 층(Layer)을 순서대로 쌓기 위한 기본 틀(모델) 생성
model.add(Dense(16, activation='relu', input_shape=(X_train_s.shape[1],)))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dense(8, activation='relu'))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dense(1, activation='sigmoid'))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])  # 모델 학습 과정(최적화 기법, 손실 함수, 평가지표) 설정
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  # 성능이 개선되지 않으면 무의미한 학습을 일찍 멈춤
model.fit(X_train_s, y_train, epochs=100, batch_size=16, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
pred_dl = (model.predict(X_test_s).flatten() > 0.5).astype(int)
print(recall_score(y_test, pred_dl))
```

</details>

In [ ]:
# 문제를 다 풀었다면 아래 셀을 실행해 소요 시간을 확인하세요.
timer.finish()

## 채점 가이드 (100점 만점 배점표)

이 모의고사는 총 9문제, 100점 만점입니다. 각 문제를 정답과 비교해 맞았으면 배점만큼, 틀렸으면 0점으로 스스로 채점해보세요.

| 문항 | 배점 |
|---|---|
| 문제 1 | 11점 |
| 문제 2 | 11점 |
| 문제 3 | 11점 |
| 문제 4 | 11점 |
| 문제 5 | 11점 |
| 문제 6 | 11점 |
| 문제 7 | 11점 |
| 문제 8 | 11점 |
| 문제 9 | 12점 |
| **합계** | **100점** |

> 💡 **합격 기준: 80점 이상** (실제 AICE Associate와 동일한 기준입니다)

### 자동으로 기록하며 채점하고 싶다면

`00_시작하기/aice_grader.py`의 `MockExamGrader`를 사용하면 점수를 자동으로 합산해줍니다.

```python
import aice_grader as grader

g = grader.MockExamGrader("모의고사06_유방암진단")
g.check(1, points=11, is_correct=True)   # 문제 1을 맞았으면 True, 틀렸으면 False
g.check(2, points=11, is_correct=True)
# ... 문제 수만큼 반복 ...
g.report()   # 최종 점수와 합격 여부를 출력
```

### 개념 체크리스트

다 풀었다면 아래 체크리스트로 한 번 더 점검해보세요.

- [ ] 라이브러리를 정확한 alias(pd, np, plt, sns 등)로 불러왔는가
- [ ] 의료 데이터에서 recall을 특히 신경 써서 확인했는가
- [ ] predict_proba로 roc_auc를 계산했는가
